Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC
# MAGIC -- Same as metabase_chat_state_rolling_15d.sql (kept filename for older links).
# MAGIC -- Metabase / Trino — Rolling 15d on first_ticket_created_date (inclusive).
# MAGIC -- Chat states UNACKNOWLEDGED | REOPENED | PENDING_RESOLUTION; latest ticket per customer in window.
# MAGIC -- Performance: prunes pl_raw_stage_tbl by event_date (reduces full-table scan timeouts).
# MAGIC
# MAGIC WITH params AS (
# MAGIC     SELECT
# MAGIC         date_add('day', -15, CURRENT_DATE) AS d_start,
# MAGIC         CURRENT_DATE AS d_end
# MAGIC ),
# MAGIC
# MAGIC latest_ticket AS (
# MAGIC     SELECT
# MAGIC         t.customer_id,
# MAGIC         t.canonical_ticket_id,
# MAGIC         t.first_ticket_created_date,
# MAGIC         t.disposition_reason,
# MAGIC         t.disposition_sub_reason,
# MAGIC         t.total_calls,
# MAGIC         t.offer_amt,
# MAGIC         t.disbursal_flag,
# MAGIC         t.rejection_flag,
# MAGIC         t.chat_ticket_state,
# MAGIC         t.pa_od_tag,
# MAGIC         t.chat_assigned_agent,
# MAGIC         t.agents_sequential,
# MAGIC         t.call_creation_timestamps,
# MAGIC         row_number() OVER (
# MAGIC             PARTITION BY t.customer_id
# MAGIC             ORDER BY t.first_ticket_created_date DESC NULLS LAST
# MAGIC         ) AS rn
# MAGIC     FROM corp_analytics.pl_apex_canonical_tickets_tbl t
# MAGIC     CROSS JOIN params p
# MAGIC     WHERE cast(t.first_ticket_created_date AS date) >= p.d_start
# MAGIC       AND cast(t.first_ticket_created_date AS date) <= p.d_end
# MAGIC ),
# MAGIC
# MAGIC apex_tickets AS (
# MAGIC     SELECT *
# MAGIC     FROM latest_ticket
# MAGIC     WHERE rn = 1
# MAGIC       AND upper(trim(cast(chat_ticket_state AS varchar))) IN (
# MAGIC           'UNACKNOWLEDGED',
# MAGIC           'REOPENED',
# MAGIC           'PENDING_RESOLUTION'
# MAGIC       )
# MAGIC ),
# MAGIC
# MAGIC eligible_customers AS (
# MAGIC     SELECT customer_id FROM apex_tickets
# MAGIC ),
# MAGIC
# MAGIC cust_bsd AS (
# MAGIC     SELECT
# MAGIC         o.customer_reference_id AS customer_id,
# MAGIC         try_cast(s.bsdv1_retrain_tier AS integer) AS bsd_tier
# MAGIC     FROM corp_analytics.customer_last_approved_offer_tbl o
# MAGIC     JOIN corp_analytics.pl_uw_scores_tiers_tbl s
# MAGIC         ON s.loan_offer_reference_id = o.loan_offer_reference_id
# MAGIC     WHERE try_cast(s.bsdv1_retrain_tier AS integer) BETWEEN 1 AND 10
# MAGIC       AND o.customer_reference_id IN (SELECT customer_id FROM eligible_customers)
# MAGIC ),
# MAGIC
# MAGIC o2d_stages AS (
# MAGIC     SELECT
# MAGIC         customer_id,
# MAGIC         date(cast(event_date AS timestamp)) AS event_dt,
# MAGIC         min(CASE WHEN stage_tag = 'OFFER_NEXT' THEN created_ts_ist END) AS offer_next_ts,
# MAGIC         min(CASE WHEN stage_tag = 'PENNY' AND coalesce(is_silent_penny, 0) = 0 THEN created_ts_ist END)
# MAGIC             AS penny_init_actual_ts,
# MAGIC         min(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'PENNY' AND status = 'SUCCESS' AND coalesce(is_silent_penny, 0) = 0
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS penny_success_actual_ts,
# MAGIC         min(CASE WHEN stage_tag = 'MANDATE' THEN created_ts_ist END) AS mandate_init_ts,
# MAGIC         min(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'MANDATE' AND status IN ('SUCCESS', 'ACTIVE', 'REGISTERED')
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS mandate_success_ts,
# MAGIC         min(CASE WHEN stage_tag = 'KYC' AND coalesce(is_kyc_replication, 0) = 0 THEN created_ts_ist END)
# MAGIC             AS kyc_success_actual_ts,
# MAGIC         min(CASE WHEN stage_tag = 'LOAN_SUMMARY_NEXT' THEN created_ts_ist END) AS loan_summary_next_ts
# MAGIC     FROM corp_analytics.pl_raw_stage_tbl
# MAGIC     CROSS JOIN params p
# MAGIC     WHERE cast(event_date AS timestamp) >= timestamp '2026-02-01 00:00:00'
# MAGIC       AND cast(event_date AS timestamp) >= date_add('day', -25, current_timestamp)
# MAGIC       AND stage_tag IN ('OFFER_NEXT', 'PENNY', 'MANDATE', 'KYC', 'LOAN_SUMMARY_NEXT')
# MAGIC       AND customer_id IN (SELECT customer_id FROM eligible_customers)
# MAGIC     GROUP BY customer_id, date(cast(event_date AS timestamp))
# MAGIC ),
# MAGIC
# MAGIC sales AS (
# MAGIC     SELECT
# MAGIC         customer_id,
# MAGIC         cast(disbursed_date AS date) AS disbursal_date
# MAGIC     FROM corp_analytics.pl_sales_fct_tbl
# MAGIC     CROSS JOIN params p
# MAGIC     WHERE cast(disbursed_date AS date) >= date '2026-02-01'
# MAGIC       AND cast(disbursed_date AS date) <= p.d_end
# MAGIC       AND customer_id IN (SELECT customer_id FROM eligible_customers)
# MAGIC ),
# MAGIC
# MAGIC o2d_post_ticket AS (
# MAGIC     SELECT
# MAGIC         a.canonical_ticket_id,
# MAGIC         s.offer_next_ts,
# MAGIC         s.penny_init_actual_ts,
# MAGIC         s.penny_success_actual_ts,
# MAGIC         s.mandate_init_ts,
# MAGIC         s.mandate_success_ts,
# MAGIC         s.kyc_success_actual_ts,
# MAGIC         s.loan_summary_next_ts
# MAGIC     FROM apex_tickets a
# MAGIC     JOIN o2d_stages s
# MAGIC         ON a.customer_id = s.customer_id
# MAGIC         AND cast(s.event_dt AS date) >= cast(a.first_ticket_created_date AS date)
# MAGIC         AND cast(s.event_dt AS date) <= date_add('day', 7, cast(a.first_ticket_created_date AS date))
# MAGIC ),
# MAGIC
# MAGIC o2d_consolidated AS (
# MAGIC     SELECT
# MAGIC         canonical_ticket_id,
# MAGIC         min(offer_next_ts) AS offer_next_ts,
# MAGIC         min(penny_init_actual_ts) AS penny_init_actual_ts,
# MAGIC         min(penny_success_actual_ts) AS penny_success_actual_ts,
# MAGIC         min(mandate_init_ts) AS mandate_init_ts,
# MAGIC         min(mandate_success_ts) AS mandate_success_ts,
# MAGIC         min(kyc_success_actual_ts) AS kyc_success_actual_ts,
# MAGIC         min(loan_summary_next_ts) AS loan_summary_next_ts
# MAGIC     FROM o2d_post_ticket
# MAGIC     GROUP BY canonical_ticket_id
# MAGIC ),
# MAGIC
# MAGIC disbursal_match AS (
# MAGIC     SELECT
# MAGIC         a.canonical_ticket_id,
# MAGIC         min(s.disbursal_date) AS funnel_disbursal_date
# MAGIC     FROM apex_tickets a
# MAGIC     JOIN sales s
# MAGIC         ON a.customer_id = s.customer_id
# MAGIC         AND cast(s.disbursal_date AS date) >= cast(a.first_ticket_created_date AS date)
# MAGIC         AND cast(s.disbursal_date AS date) <= date_add('day', 7, cast(a.first_ticket_created_date AS date))
# MAGIC     GROUP BY a.canonical_ticket_id
# MAGIC )
# MAGIC
# MAGIC SELECT
# MAGIC     cast(a.customer_id AS varchar) AS customer_id,
# MAGIC     cast(coalesce(cast(a.canonical_ticket_id AS varchar), '') AS varchar) AS canonical_ticket_id,
# MAGIC     coalesce(cast(a.first_ticket_created_date AS timestamp), timestamp '1900-01-01 00:00:00') AS first_ticket_created_date,
# MAGIC     coalesce(
# MAGIC         date_diff('day', cast(a.first_ticket_created_date AS date), current_date),
# MAGIC         0
# MAGIC     ) AS days_in_pipeline,
# MAGIC     cast(coalesce(a.disposition_reason, 'UNKNOWN') AS varchar) AS disposition_reason,
# MAGIC     cast(coalesce(a.disposition_sub_reason, '') AS varchar) AS disposition_sub_reason,
# MAGIC     coalesce(a.total_calls, 0) AS total_calls,
# MAGIC     coalesce(a.offer_amt, 0) AS offer_amt,
# MAGIC     coalesce(a.disbursal_flag, 0) AS disbursal_flag,
# MAGIC     coalesce(a.rejection_flag, 0) AS rejection_flag,
# MAGIC     cast(coalesce(a.chat_ticket_state, '') AS varchar) AS chat_ticket_state,
# MAGIC     cast(coalesce(a.pa_od_tag, '') AS varchar) AS pa_od_tag,
# MAGIC     cast(coalesce(a.chat_assigned_agent, '') AS varchar) AS chat_assigned_agent,
# MAGIC     cast(coalesce(a.agents_sequential, '[]') AS varchar) AS agents_sequential,
# MAGIC     cast(coalesce(a.call_creation_timestamps, '[]') AS varchar) AS call_creation_timestamps,
# MAGIC     coalesce(try_cast(cb.bsd_tier AS integer), -1) AS bsd_tier,
# MAGIC     cast(
# MAGIC         CASE
# MAGIC             WHEN d.funnel_disbursal_date IS NOT NULL THEN 'DISBURSED'
# MAGIC             WHEN o.loan_summary_next_ts IS NOT NULL THEN 'LOAN_SUMMARY'
# MAGIC             WHEN o.kyc_success_actual_ts IS NOT NULL THEN 'KYC_SUCCESS'
# MAGIC             WHEN o.mandate_success_ts IS NOT NULL THEN 'MANDATE_SUCCESS'
# MAGIC             WHEN o.mandate_init_ts IS NOT NULL THEN 'MANDATE_INIT'
# MAGIC             WHEN o.penny_success_actual_ts IS NOT NULL THEN 'PENNY_SUCCESS'
# MAGIC             WHEN o.penny_init_actual_ts IS NOT NULL THEN 'PENNY_INIT'
# MAGIC             WHEN o.offer_next_ts IS NOT NULL THEN 'OFFER_NEXT'
# MAGIC             ELSE 'OFFER_ONLY'
# MAGIC         END AS varchar
# MAGIC     ) AS furthest_stage,
# MAGIC     CASE upper(trim(cast(a.chat_ticket_state AS varchar)))
# MAGIC         WHEN 'UNACKNOWLEDGED' THEN 1
# MAGIC         WHEN 'REOPENED' THEN 2
# MAGIC         WHEN 'PENDING_RESOLUTION' THEN 3
# MAGIC         ELSE 99
# MAGIC     END AS state_call_priority,
# MAGIC     row_number() OVER (
# MAGIC         ORDER BY
# MAGIC             CASE upper(trim(cast(a.chat_ticket_state AS varchar)))
# MAGIC                 WHEN 'UNACKNOWLEDGED' THEN 1
# MAGIC                 WHEN 'REOPENED' THEN 2
# MAGIC                 WHEN 'PENDING_RESOLUTION' THEN 3
# MAGIC                 ELSE 99
# MAGIC             END ASC,
# MAGIC             coalesce(try_cast(cb.bsd_tier AS integer), -1) DESC,
# MAGIC             cast(a.first_ticket_created_date AS timestamp) ASC
# MAGIC     ) AS call_queue_rank
# MAGIC FROM apex_tickets a
# MAGIC LEFT JOIN cust_bsd cb ON cb.customer_id = a.customer_id
# MAGIC LEFT JOIN o2d_consolidated o ON o.canonical_ticket_id = a.canonical_ticket_id
# MAGIC LEFT JOIN disbursal_match d ON d.canonical_ticket_id = a.canonical_ticket_id
# MAGIC ORDER BY call_queue_rank
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url =  "https://docs.google.com/spreadsheets/d/1iWBV4WBiS_Gq1SYIxIzbjXb9bBEy0maZ5mV5I2g-lnY/edit"  # New V4
worksheet_name = "Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)